In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
# Load all 14 files into a dictionary — work from this throughout
files = [
    "circuits.csv", "constructor_results.csv", "constructor_standings.csv",
    "constructors.csv", "driver_standings.csv", "drivers.csv",
    "lap_times.csv", "pit_stops.csv", "qualifying.csv",
    "races.csv", "results.csv", "seasons.csv",
    "sprint_results.csv", "status.csv"
]

dfs = {}
for f in files:
    name = f.replace(".csv", "")
    dfs[name] = pd.read_csv(f"../dataset/{f}", na_values=['\\N', '', 'NA'])
    print(f"Loaded {name}: {dfs[name].shape}")

Loaded circuits: (78, 9)
Loaded constructor_results: (12898, 5)
Loaded constructor_standings: (13664, 7)
Loaded constructors: (214, 5)
Loaded driver_standings: (35427, 7)
Loaded drivers: (865, 9)
Loaded lap_times: (618766, 6)
Loaded pit_stops: (22193, 7)
Loaded qualifying: (11036, 9)
Loaded races: (1171, 18)
Loaded results: (27304, 18)
Loaded seasons: (77, 2)
Loaded sprint_results: (502, 17)
Loaded status: (140, 2)


In [5]:
for name, df in dfs.items():
    print(f"\n{'='*40}")
    print(f"TABLE: {name}")
    print(f"{'='*40}")
    print(df.dtypes)


TABLE: circuits
circuitId       int64
circuitRef     object
name           object
location       object
country        object
lat           float64
lng           float64
alt             int64
url            object
dtype: object

TABLE: constructor_results
constructorResultsId      int64
raceId                    int64
constructorId             int64
points                  float64
status                   object
dtype: object

TABLE: constructor_standings
constructorStandingsId      int64
raceId                      int64
constructorId               int64
points                    float64
position                  float64
positionText               object
wins                        int64
dtype: object

TABLE: constructors
constructorId      int64
constructorRef    object
name              object
nationality       object
url               object
dtype: object

TABLE: driver_standings
driverStandingsId      int64
raceId                 int64
driverId               int64
points         

# Casting Date Columns

## Casting Date Columns to datetime64

In [6]:
# Race Table

# Cast it
dfs['races']['date'] = pd.to_datetime(dfs['races']['date'], errors='coerce')

# Verify — should show datetime64[ns] not object
print(dfs['races']['date'].dtype)

# Check a sample
print(dfs['races']['date'].head())

datetime64[ns]
0   2009-03-29
1   2009-04-05
2   2009-04-19
3   2009-04-26
4   2009-05-10
Name: date, dtype: datetime64[ns]


In [14]:
# Verify all date casts worked
print("=== DATE COLUMN VERIFICATION ===\n")

# Check races date columns
races_date_cols = ['date', 'fp1_date', 'fp2_date', 'fp3_date', 'quali_date', 'sprint_date']
print("RACES:")
for col in races_date_cols:
    print(f"  {col:<15} {str(dfs['races'][col].dtype):<20} nulls: {dfs['races'][col].isnull().sum()}")

# Check drivers dob
print("\nDRIVERS:")
print(f"  {'dob':<15} {str(dfs['drivers']['dob'].dtype):<20} nulls: {dfs['drivers']['dob'].isnull().sum()}")

# Check no object types remain for date columns
print("\n=== ANY MISSED DATE COLUMNS? ===")
for name, df in dfs.items():
    for col in df.columns:
        if 'date' in col.lower() or col == 'dob':
            if df[col].dtype == 'object':
                print(f"  ❌ {name}.{col} still object")
print("  ✅ Done")


=== DATE COLUMN VERIFICATION ===

RACES:
  date            datetime64[ns]       nulls: 0
  fp1_date        datetime64[ns]       nulls: 1035
  fp2_date        datetime64[ns]       nulls: 1053
  fp3_date        datetime64[ns]       nulls: 1065
  quali_date      datetime64[ns]       nulls: 1035
  sprint_date     datetime64[ns]       nulls: 1141

DRIVERS:
  dob             datetime64[ns]       nulls: 0

=== ANY MISSED DATE COLUMNS? ===
  ✅ Done


NOTE:

Two questions:

Q1 — fp1_date worked but fp2_date, fp3_date etc. didn't — even though they're all in the same loop. And date worked but it's not even in your list. That's strange. What does this tell you about whether your cast cell actually ran, versus what's happening?


Q2 — Here's the big clue: did you run your cells in order? In Jupyter, if you run cells out of order, or ran an old version of the cell, the state gets messy.

In [15]:
# ── CAST + VERIFY DATE COLUMNS (run this single cell) ─────────────────────────
print("=== CASTING DATE COLUMNS ===\n")

# All date columns that need casting
races_date_cols = ['date', 'fp1_date', 'fp2_date', 'fp3_date', 'quali_date', 'sprint_date']

# Cast every races date column
for col in races_date_cols:
    dfs['races'][col] = pd.to_datetime(dfs['races'][col], errors='coerce')
    print(f"  races.{col:<15} → {dfs['races'][col].dtype}")

# Cast drivers dob
dfs['drivers']['dob'] = pd.to_datetime(dfs['drivers']['dob'], errors='coerce')
print(f"  drivers.{'dob':<13} → {dfs['drivers']['dob'].dtype}")


# ── VERIFICATION ──────────────────────────────────────────────────────────────
print("\n=== DATE COLUMN VERIFICATION ===\n")

print("RACES:")
for col in races_date_cols:
    dtype = str(dfs['races'][col].dtype)
    nulls = dfs['races'][col].isnull().sum()
    status = "✅" if dtype.startswith("datetime") else "❌"
    print(f"  {status} {col:<15} {dtype:<20} nulls: {nulls}")

print("\nDRIVERS:")
dtype = str(dfs['drivers']['dob'].dtype)
nulls = dfs['drivers']['dob'].isnull().sum()
status = "✅" if dtype.startswith("datetime") else "❌"
print(f"  {status} {'dob':<15} {dtype:<20} nulls: {nulls}")


# ── FINAL CHECK — ANY DATE COLUMN STILL OBJECT? ──────────────────────────────
print("\n=== ANY MISSED DATE COLUMNS? ===")
missed = False
for name, df in dfs.items():
    for col in df.columns:
        if 'date' in col.lower() or col == 'dob':
            if df[col].dtype == 'object':
                print(f"  ❌ {name}.{col} still object")
                missed = True
if not missed:
    print("  ✅ All date columns successfully cast")

=== CASTING DATE COLUMNS ===

  races.date            → datetime64[ns]
  races.fp1_date        → datetime64[ns]
  races.fp2_date        → datetime64[ns]
  races.fp3_date        → datetime64[ns]
  races.quali_date      → datetime64[ns]
  races.sprint_date     → datetime64[ns]
  drivers.dob           → datetime64[ns]

=== DATE COLUMN VERIFICATION ===

RACES:
  ✅ date            datetime64[ns]       nulls: 0
  ✅ fp1_date        datetime64[ns]       nulls: 1035
  ✅ fp2_date        datetime64[ns]       nulls: 1053
  ✅ fp3_date        datetime64[ns]       nulls: 1065
  ✅ quali_date      datetime64[ns]       nulls: 1035
  ✅ sprint_date     datetime64[ns]       nulls: 1141

DRIVERS:
  ✅ dob             datetime64[ns]       nulls: 0

=== ANY MISSED DATE COLUMNS? ===
  ✅ All date columns successfully cast


# Parse Time Strings to Milliseconds

## Task 3 — Parse Time Strings to Milliseconds

### The Problem

These columns hold times as text:

| Column | Example | Meaning |
|--------|---------|---------|
| `lap_times.time` | `"1:27.452"` | 1 min 27.452 sec |
| `qualifying.q1` / `q2` / `q3` | `"1:23.567"` | qualifying lap time |
| `results.fastestLapTime` | `"1:25.123"` | fastest lap of the race |

You can't average or compare `"1:27.452"` as a string.
You need it as a number: **87452 milliseconds**.

---

### Logic — Worked Out Before Coding

**Q1 — Breaking down `"1:27.452"` (format `M:SS.mmm`)**

- Minutes = `1`
- Seconds = `27`
- Milliseconds = `452`

**Q2 — The maths**

"1:27.452"  →  (1 × 60000) + (27 × 1000) + 452

→   60000 + 27000 + 452

→   87452 ms

General formula:

total_ms = (minutes × 60000) + (seconds × 1000) + milliseconds

**Q3 — Handling nulls**

- If the input is `NaN` / null → return `NaN`
- Do not return `0` — that would falsely imply a lap time of zero
- Null must stay null to preserve structural and logical gaps


In [16]:
# Tests
test = "1:27.452"

# Split on the colon
parts = test.split(":")
print(parts)

['1', '27.452']


In [17]:
test = "1:27.452"
parts = test.split(":")

minutes = float(parts[0])
seconds = float(parts[1])

print("minutes:", minutes)
print("seconds:", seconds)

minutes: 1.0
seconds: 27.452


In [18]:
total_ms = (minutes * 60000) + (seconds * 1000)   # your formula here

print(total_ms)  # should print 87452.0

87452.0


In [36]:
def time_to_ms(time_str):
    if pd.isna(time_str):
        return np.nan

    parts = time_str.split(":")

    if len(parts) == 1:
        # plain seconds, no colon — e.g. "49.111"
        seconds = float(parts[0])
        total_ms = (seconds * 1000)

    elif len(parts) == 2:
        minutes = float(parts[0])
        seconds = float(parts[1])
        total_ms = (minutes * 60000) + (seconds * 1000)

    elif len(parts) == 3:
        hours   = float(parts[0])
        minutes = float(parts[1])
        seconds = float(parts[2])
        total_ms = (hours * 3600000) + (minutes * 60000) + (seconds * 1000)

    return int(total_ms)

In [37]:
import numpy as np  # make sure this is imported

# Test cases
tests = ["1:27.452", "0:58.123", "1:23.567", np.nan, "2:05.999"]

for t in tests:
    print(f"{str(t):<12} → {time_to_ms(t)}")

1:27.452     → 87452
0:58.123     → 58123
1:23.567     → 83567
nan          → nan
2:05.999     → 125999


In [38]:
# Apply to lap_times.time — create a new ms column
dfs['lap_times']['time_ms'] = dfs['lap_times']['time'].apply(time_to_ms)

# Verify
print(dfs['lap_times'][['time', 'time_ms']].head(10))
print(f"\ndtype: {dfs['lap_times']['time_ms'].dtype}")
print(f"nulls: {dfs['lap_times']['time_ms'].isnull().sum()}")

       time  time_ms
0  1:38.109    98109
1  1:33.006    93006
2  1:32.713    92713
3  1:32.803    92803
4  1:32.342    92342
5  1:32.605    92605
6  1:32.502    92502
7  1:32.537    92537
8  1:33.240    93240
9  1:32.572    92572

dtype: int64
nulls: 0


In [39]:
# Compare your parsed values against the existing milliseconds column
mismatches = dfs['lap_times'][dfs['lap_times']['time_ms'] != dfs['lap_times']['milliseconds']]

print(f"Total rows:    {len(dfs['lap_times']):,}")
print(f"Mismatches:    {len(mismatches):,}")

if len(mismatches) == 0:
    print("\n✅ PERFECT — your function matches the official milliseconds exactly")
else:
    print("\n⚠️ Some mismatches — let's look:")
    print(mismatches[['time', 'time_ms', 'milliseconds']].head())

Total rows:    618,766
Mismatches:    0

✅ PERFECT — your function matches the official milliseconds exactly


In [48]:
# If there are mismatches, investigate a few to see if it's a parsing issue or maybe an error in the original milliseconds column
# qualifying — three columns
dfs['qualifying']['q1_ms'] = dfs['qualifying']['q1'].apply(time_to_ms)
dfs['qualifying']['q2_ms'] = dfs['qualifying']['q2'].apply(time_to_ms)
dfs['qualifying']['q3_ms'] = dfs['qualifying']['q3'].apply(time_to_ms)

# results
dfs['results']['fastestLapTime_ms'] = dfs['results']['fastestLapTime'].apply(time_to_ms)

# sprint_results
dfs['sprint_results']['fastestLapTime_ms'] = dfs['sprint_results']['fastestLapTime'].apply(time_to_ms)

# pit_stops
dfs['pit_stops']['duration_ms'] = dfs['pit_stops']['duration'].apply(time_to_ms)

# Verify nulls preserved (not turned into 0)
print("qualifying q1:", dfs['qualifying']['q1'].isnull().sum(), "→ q1_ms:", dfs['qualifying']['q1_ms'].isnull().sum())
print("qualifying q2:", dfs['qualifying']['q2'].isnull().sum(), "→ q2_ms:", dfs['qualifying']['q2_ms'].isnull().sum())
print("qualifying q3:", dfs['qualifying']['q3'].isnull().sum(), "→ q3_ms:", dfs['qualifying']['q3_ms'].isnull().sum())
print("results fLap:", dfs['results']['fastestLapTime'].isnull().sum(), "→ ms:", dfs['results']['fastestLapTime_ms'].isnull().sum())
print("sprint fLap:", dfs['sprint_results']['fastestLapTime'].isnull().sum(), "→ ms:", dfs['sprint_results']['fastestLapTime_ms'].isnull().sum())

# Verify some samples
print(dfs['pit_stops'][['duration', 'duration_ms']].head(10))
print(f"\ndtype: {dfs['pit_stops']['duration_ms'].dtype}")
print(f"original nulls: {dfs['pit_stops']['duration'].isnull().sum()}")
print(f"new nulls:      {dfs['pit_stops']['duration_ms'].isnull().sum()}")

qualifying q1: 163 → q1_ms: 163
qualifying q2: 4787 → q2_ms: 4787
qualifying q3: 7143 → q3_ms: 7143
results fLap: 18535 → ms: 18535
sprint fLap: 13 → ms: 13
  duration  duration_ms
0   49.111      49111.0
1   28.482      28482.0
2   43.745      43745.0
3   21.992      21992.0
4   27.693      27693.0
5   24.806      24806.0
6    53.89      53890.0
7   34.437      34437.0
8   23.342      23342.0
9   56.753      56753.0

dtype: float64
original nulls: 3
new nulls:      3


In [49]:
for name, df in dfs.items():
    for col in df.select_dtypes(include='float64').columns:
        non_null = df[col].dropna()
        if len(non_null) > 0 and (non_null == non_null.astype(int)).all():
            print(f"{name}.{col}")

constructor_standings.position
driver_standings.position
drivers.number
pit_stops.milliseconds
pit_stops.duration_ms
qualifying.q1_ms
qualifying.q2_ms
qualifying.q3_ms
results.number
results.grid
results.position
results.milliseconds
results.fastestLap
results.rank
results.fastestLapTime_ms
sprint_results.position
sprint_results.milliseconds
sprint_results.fastestLap
sprint_results.rank
sprint_results.fastestLapTime_ms


In [51]:
int_cols = {
    'constructor_standings': ['position'],
    'driver_standings': ['position'],
    'drivers': ['number'],
    'pit_stops': ['milliseconds', 'duration_ms'],
    'qualifying': ['q1_ms', 'q2_ms', 'q3_ms'],
    'results': ['number', 'grid', 'position', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime_ms'],          # fill in all results columns
    'sprint_results': ['position', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime_ms'],   # fill in all sprint_results columns
}

In [52]:
for table, cols in int_cols.items():
    for col in cols:
        dfs[table][col] = dfs[table][col].astype('Int64')
        print(f"  {table}.{col:<20} → {dfs[table][col].dtype}")

  constructor_standings.position             → Int64
  driver_standings.position             → Int64
  drivers.number               → Int64
  pit_stops.milliseconds         → Int64
  pit_stops.duration_ms          → Int64
  qualifying.q1_ms                → Int64
  qualifying.q2_ms                → Int64
  qualifying.q3_ms                → Int64
  results.number               → Int64
  results.grid                 → Int64
  results.position             → Int64
  results.milliseconds         → Int64
  results.fastestLap           → Int64
  results.rank                 → Int64
  results.fastestLapTime_ms    → Int64
  sprint_results.position             → Int64
  sprint_results.milliseconds         → Int64
  sprint_results.fastestLap           → Int64
  sprint_results.rank                 → Int64
  sprint_results.fastestLapTime_ms    → Int64


In [53]:
import os

# New folder for cleaned data — keeps raw dataset/ untouched
os.makedirs('../data_cleaned', exist_ok=True)
print("Folder ready")

Folder ready


In [60]:
import sys
print(sys.executable)   # ← this tells us WHICH python your notebook uses

/usr/local/bin/python3


In [68]:
import os
os.makedirs('../data_cleaned', exist_ok=True)

# Save all dataframes as pickle
for name, df in dfs.items():
    path = f'../data_cleaned/{name}.pkl'
    df.to_pickle(path)
    print(f"  saved {name:<25} → {path}")

  saved circuits                  → ../data_cleaned/circuits.pkl
  saved constructor_results       → ../data_cleaned/constructor_results.pkl
  saved constructor_standings     → ../data_cleaned/constructor_standings.pkl
  saved constructors              → ../data_cleaned/constructors.pkl
  saved driver_standings          → ../data_cleaned/driver_standings.pkl
  saved drivers                   → ../data_cleaned/drivers.pkl
  saved lap_times                 → ../data_cleaned/lap_times.pkl
  saved pit_stops                 → ../data_cleaned/pit_stops.pkl
  saved qualifying                → ../data_cleaned/qualifying.pkl
  saved races                     → ../data_cleaned/races.pkl
  saved results                   → ../data_cleaned/results.pkl
  saved seasons                   → ../data_cleaned/seasons.pkl
  saved sprint_results            → ../data_cleaned/sprint_results.pkl
  saved status                    → ../data_cleaned/status.pkl


In [69]:
import pandas as pd
test = pd.read_pickle('../data_cleaned/lap_times.pkl')
print(test.dtypes)
print(f"\nrows: {len(test):,}")

raceId           int64
driverId         int64
lap              int64
position         int64
time            object
milliseconds     int64
time_ms          int64
dtype: object

rows: 618,766
